In [59]:
from datetime import date, datetime, timedelta
import pandas as pd
import numpy as np
import weather_data as wd
from mlflow import mlflow, MlflowClient
import pickle
import xgboost  as xgb

In [37]:
current_dateTime = datetime.now()
#current_dateTime= datetime(2026,8,3,15,30)
#current_dateTime= datetime(2025,8,28,6,00)
current_date=current_dateTime.strftime("%Y-%m-%d")
current_hour=int(current_dateTime.strftime("%H"))
current_minute=int(current_dateTime.strftime("%M"))
weather_forecast_hour = current_hour if current_minute<30 else (current_hour+1)
weather_forecast_time=f"{weather_forecast_hour:02}:00:00"
current_date, weather_forecast_time

('2026-02-09', '17:00:00')

In [38]:
current_date, current_minute, weather_forecast_time, 

('2026-02-09', 25, '17:00:00')

In [40]:
latitude=45.704178167781265
longitude=9.663174137924273

timezone="Europe/Berlin"

In [41]:
df_hourly=wd.get_weather_data(start_date=current_date, end_date=current_date,latitude=latitude,longitude=longitude, timezone=timezone)
df_hourly

,timestamp,weather_code,cloud_cover,temperature
0,2026-02-08 23:00:00,3.0,22.0,5.8685
1,2026-02-09 00:00:00,3.0,42.0,5.4685
2,2026-02-09 01:00:00,3.0,98.0,5.4685
3,2026-02-09 02:00:00,3.0,48.0,5.3685
4,2026-02-09 03:00:00,3.0,11.0,5.3685
5,2026-02-09 04:00:00,3.0,58.0,5.9685
6,2026-02-09 05:00:00,3.0,0.0,6.0185
7,2026-02-09 06:00:00,3.0,0.0,5.8185
8,2026-02-09 07:00:00,3.0,9.0,6.2685
9,2026-02-09 08:00:00,3.0,0.0,7.0185


In [42]:
df_eval=pd.DataFrame()
weather_fields=["weather_code","cloud_cover","temperature"]
df_eval["time"]=df_hourly.timestamp.dt.time
df_eval["month"] = current_dateTime.month
df_eval[weather_fields] = df_hourly[weather_fields]
df_eval

,time,month,weather_code,cloud_cover,temperature
0,23:00:00,2,3.0,22.0,5.8685
1,00:00:00,2,3.0,42.0,5.4685
2,01:00:00,2,3.0,98.0,5.4685
3,02:00:00,2,3.0,48.0,5.3685
4,03:00:00,2,3.0,11.0,5.3685
5,04:00:00,2,3.0,58.0,5.9685
6,05:00:00,2,3.0,0.0,6.0185
7,06:00:00,2,3.0,0.0,5.8185
8,07:00:00,2,3.0,9.0,6.2685
9,08:00:00,2,3.0,0.0,7.0185


In [43]:

df_eval = df_eval[ ( (df_eval.time.astype(str)>=weather_forecast_time) & (df_eval.time.astype(str)<'20:30:00') ) ]
df_eval, weather_forecast_time


(        time  month  weather_code  cloud_cover  temperature
 18  17:00:00      2           3.0         28.0       8.9685
 19  18:00:00      2          80.0         87.0       8.3685
 20  19:00:00      2           3.0          6.0       7.7685
 21  20:00:00      2           3.0         18.0       7.3685,
 '17:00:00')

In [44]:
myrow = df_eval.iloc[0]
type(myrow["time"])
dt = datetime.combine(date.today(), myrow["time"]) + timedelta(minutes=30)
print (dt.time())
#myrow["time"] = myrow["time"] + timedelta(minutes=30)
#myrow

17:30:00


In [45]:
df_eval

,time,month,weather_code,cloud_cover,temperature
18,17:00:00,2,3.0,28.0,8.9685
19,18:00:00,2,80.0,87.0,8.3685
20,19:00:00,2,3.0,6.0,7.7685
21,20:00:00,2,3.0,18.0,7.3685


In [46]:
# Replicate weather data at half-hours to match model data
df_eval_30 = df_eval.copy()
today = date.today()
for idx, row in df_eval_30.iterrows():
    dt = datetime.combine(today, row["time"]) + timedelta(minutes=30)
    df_eval_30.at[idx,"time"] = dt.time()

df_eval = pd.concat([df_eval, df_eval_30]).sort_values('time')
df_eval


,time,month,weather_code,cloud_cover,temperature
18,17:00:00,2,3.0,28.0,8.9685
18,17:30:00,2,3.0,28.0,8.9685
19,18:00:00,2,80.0,87.0,8.3685
19,18:30:00,2,80.0,87.0,8.3685
20,19:00:00,2,3.0,6.0,7.7685
20,19:30:00,2,3.0,6.0,7.7685
21,20:00:00,2,3.0,18.0,7.3685
21,20:30:00,2,3.0,18.0,7.3685


In [49]:
mlflow_tracking_uri = "http://localhost:5000"
experiment_name="solar-surplus-estimator-experiment"

mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment(experiment_name)

runs = mlflow.search_runs(filter_string="status='FINISHED'")
#df_runs = client.search_runs(experiment_ids=experiment_name,filter_string="status='FINISHED'")
#df_runs = client.search_runs(experiment_ids=experiment_name)
runs


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.best_iteration,metrics.rmse_test,metrics.stopped_iteration,metrics.validation-rmse,...,params.custom_metric,params.learning_rate,params.min_child_weight,params.reg_alpha,tags.mlflow.source.type,tags.model_type,tags.mlflow.source.git.commit,tags.mlflow.user,tags.mlflow.source.name,tags.mlflow.runName
0,7459a5bee0c84caca7ac47cf1c603907,2,FINISHED,mlflow-artifacts:/2/7459a5bee0c84caca7ac47cf1c...,2026-02-09 16:22:19.126000+00:00,2026-02-09 16:22:53.274000+00:00,4.0,425.039303,54.0,398.278795,...,None,0.21661675016798473,0.44831560205900217,0.011200550717078536,LOCAL,xgboost,3f8236fa2cdd55596c00d00bcc66ca0191ef1079,calinet,solar-estimator-model-pipeline.py,mercurial-loon-688
1,34b1cf7629514ff3acb4e77d75882327,2,FINISHED,mlflow-artifacts:/2/34b1cf7629514ff3acb4e77d75...,2026-02-09 16:21:26.922000+00:00,2026-02-09 16:22:19.066000+00:00,199.0,328.910834,249.0,328.778405,...,None,0.12020347637135255,4.234532236734746,0.18476603949102705,LOCAL,xgboost,3f8236fa2cdd55596c00d00bcc66ca0191ef1079,calinet,solar-estimator-model-pipeline.py,mercurial-ant-365
2,f906acace8db46e284c02881e2c5df19,2,FINISHED,mlflow-artifacts:/2/f906acace8db46e284c02881e2...,2026-02-09 16:20:53.547000+00:00,2026-02-09 16:21:26.858000+00:00,3.0,393.037250,53.0,377.097838,...,None,0.3172531391242561,0.8219337035122293,0.08408850386674233,LOCAL,xgboost,3f8236fa2cdd55596c00d00bcc66ca0191ef1079,calinet,solar-estimator-model-pipeline.py,peaceful-bear-678
3,c3886647ff344c67bf0828fcca18e71b,2,FINISHED,mlflow-artifacts:/2/c3886647ff344c67bf0828fcca...,2026-02-09 16:20:42.367000+00:00,2026-02-09 16:20:53.487000+00:00,1.0,358.288144,51.0,345.042170,...,None,0.7441178408725314,10.910615785527055,0.3067494048571722,LOCAL,xgboost,3f8236fa2cdd55596c00d00bcc66ca0191ef1079,calinet,solar-estimator-model-pipeline.py,victorious-bass-406
4,1f3e3ed32c9546acab7790e467c07a4f,2,FINISHED,mlflow-artifacts:/2/1f3e3ed32c9546acab7790e467...,2026-02-09 16:20:19.928000+00:00,2026-02-09 16:20:42.305000+00:00,27.0,330.107504,77.0,329.272945,...,None,0.20189167602257016,15.797299108873343,0.047273735495560756,LOCAL,xgboost,3f8236fa2cdd55596c00d00bcc66ca0191ef1079,calinet,solar-estimator-model-pipeline.py,receptive-calf-750
5,dcdf0f2c445c4f92bb329f104ec4d483,2,FINISHED,mlflow-artifacts:/2/dcdf0f2c445c4f92bb329f104e...,2026-02-09 16:19:52.164000+00:00,2026-02-09 16:20:19.867000+00:00,93.0,334.980394,143.0,334.237317,...,None,0.18261702035215893,1.9619978867168812,0.11785830506042831,LOCAL,xgboost,3f8236fa2cdd55596c00d00bcc66ca0191ef1079,calinet,solar-estimator-model-pipeline.py,burly-skink-676
6,fd0db12ccd6a4012b2c076b77cebfed5,2,FINISHED,mlflow-artifacts:/2/fd0db12ccd6a4012b2c076b77c...,2026-02-09 16:19:41.201000+00:00,2026-02-09 16:19:52.096000+00:00,199.0,337.088960,249.0,335.749964,...,None,0.7717743158265616,3.023121968372186,0.036989958603775,LOCAL,xgboost,3f8236fa2cdd55596c00d00bcc66ca0191ef1079,calinet,solar-estimator-model-pipeline.py,receptive-pug-748
7,92f846cbaf0a4065a5291a98dbafdbd8,2,FINISHED,mlflow-artifacts:/2/92f846cbaf0a4065a5291a98db...,2026-02-09 16:18:59.557000+00:00,2026-02-09 16:19:41.140000+00:00,87.0,329.323801,137.0,328.884559,...,None,0.08098509174456076,10.006624357582758,0.0070524965267841164,LOCAL,xgboost,3f8236fa2cdd55596c00d00bcc66ca0191ef1079,calinet,solar-estimator-model-pipeline.py,likeable-panda-821
8,79a1e369d4ff4b26b9b5b201528ebbb8,2,FINISHED,mlflow-artifacts:/2/79a1e369d4ff4b26b9b5b20152...,2026-02-09 16:18:30.628000+00:00,2026-02-09 16:18:59.496000+00:00,204.0,323.670099,254.0,323.579000,...,None,0.1894803024857166,13.665314561131234,0.01845622640991709,LOCAL,xgboost,3f8236fa2cdd55596c00d00bcc66ca0191ef1079,calinet,solar-estimator-model-pipeline.py,orderly-mouse-217
9,5fdedfde237344aabef2b713d1916ca1,2,FINISHED,mlflow-artifacts:/2/5fdedfde237344aabef2b713d1...,2026-02-09 16:17:50.070000+00:00,2026-02-09 16:18:30.567000+00:00,106.0,336.893116,156.0,336.621967,...,None,0.10249101427398996,6.95475

In [50]:
best_run = runs.loc[ runs['metrics.rmse_test'].idxmin() ]
best_run_id = best_run['run_id']
best_run_rmse = best_run['metrics.rmse_test']
print(f'best run_id: {best_run_id}, rmse: {best_run_rmse}')
best_run, best_run.artifact_uri

best run_id: 5781a87702dd4bdd9bb428b8708ac8ca, rmse: 321.4108229940825


(run_id                                            5781a87702dd4bdd9bb428b8708ac8ca
 experiment_id                                                                    2
 status                                                                    FINISHED
 artifact_uri                     mlflow-artifacts:/2/5781a87702dd4bdd9bb428b870...
 start_time                                        2026-02-09 15:24:07.229000+00:00
 end_time                                          2026-02-09 15:24:17.991000+00:00
 metrics.best_iteration                                                       155.0
 metrics.rmse_test                                                       321.410823
 metrics.stopped_iteration                                                    205.0
 metrics.validation-rmse                                                 320.969303
 params.num_boost_round                                                        1000
 params.max_depth                                                           

In [51]:
client = MlflowClient(mlflow_tracking_uri)
 
# Download run artifact
path = client.download_artifacts(run_id=best_run_id, path="preprocessor/preprocessor.b")

with open(path, 'rb') as f_out:
   dv = pickle.load(f_out)


In [52]:
model_filter_string=f"source_run_id='{best_run_id}'"
models = mlflow.search_logged_models(filter_string=model_filter_string)
best_model_id=models.at[0,"model_id"]
best_model_id

'm-e1067b014dfc4fe49f11a3df7f087fe8'

In [53]:
#filter_string=f"run_id='{best_run_id}'"
#model_name="solar-energy-surplus-predictor"
#model_id="m-39519ca974a7412281c395e355db9252"
#filter_string="name='solar-energy-surplus-predictor'"
#model_version=mlflow.search_model_versions(filter_string=filter_string)
#model_version=mlflow.search_model_versions()
#filter_string = f"name='{model_name}'"
#results = client.search_model_versions()
#logged_model=f"runs:/{best_run_id}/models/"
logged_model=f"models:/{best_model_id}"
#runs:/<mlflow_run_id>/run-relative/path/to/model
booster = mlflow.xgboost.load_model(logged_model)
# model_data = client.get_logged_model(logged_model)
# model = joblib.load(model_data)
best_run_id,  booster
#logged_model

('5781a87702dd4bdd9bb428b8708ac8ca', <xgboost.core.Booster at 0x70c50022e210>)

In [54]:
# Set features
categorical=['time', 'month','weather_code', 'cloud_cover']
numerical=[ 'temperature' ]

df_eval[categorical] = df_eval[categorical].astype(str)

X_eval_dicts = df_eval[categorical+numerical].to_dict(orient='records')


In [55]:
# Eval row by row
for row in X_eval_dicts:
    print(row)
    X_eval = dv.transform(row)
    X_eval_dm = xgb.DMatrix(X_eval)
    y_pred = booster.predict(X_eval_dm)
    print(y_pred)   

#X_eval = dv.transform(X_eval_dicts)

# X_eval_dm = xgb.DMatrix(X_eval)

# y_pred = booster.predict(X_eval_dm)
# y_pred



{'time': '17:00:00', 'month': '2', 'weather_code': '3.0', 'cloud_cover': '28.0', 'temperature': 8.968500137329102}
[-261.99472]
{'time': '17:30:00', 'month': '2', 'weather_code': '3.0', 'cloud_cover': '28.0', 'temperature': 8.968500137329102}
[-189.6828]
{'time': '18:00:00', 'month': '2', 'weather_code': '80.0', 'cloud_cover': '87.0', 'temperature': 8.368499755859375}
[43.412273]
{'time': '18:30:00', 'month': '2', 'weather_code': '80.0', 'cloud_cover': '87.0', 'temperature': 8.368499755859375}
[93.279434]
{'time': '19:00:00', 'month': '2', 'weather_code': '3.0', 'cloud_cover': '6.0', 'temperature': 7.768500328063965}
[-29.158184]
{'time': '19:30:00', 'month': '2', 'weather_code': '3.0', 'cloud_cover': '6.0', 'temperature': 7.768500328063965}
[-48.9017]
{'time': '20:00:00', 'month': '2', 'weather_code': '3.0', 'cloud_cover': '18.0', 'temperature': 7.368499755859375}
[-6.205317]
{'time': '20:30:00', 'month': '2', 'weather_code': '3.0', 'cloud_cover': '18.0', 'temperature': 7.368499755859

In [56]:
# Eval all rows together
X_eval = dv.transform(X_eval_dicts)
X_eval_dm = xgb.DMatrix(X_eval)
y_pred = booster.predict(X_eval_dm)
print(y_pred)

[-261.99472  -189.6828     43.412273   93.279434  -29.158184  -48.9017
   -6.205317   17.38041 ]


In [ ]:
# Remove negative surplus predictions
z = np.zeros(len(y_pred))
y_pred_abs = np.maximum(y_pred,z)
y_pred_abs


array([ 0.        ,  0.        , 43.41227341, 93.2794342 ,  0.        ,
        0.        ,  0.        , 17.38040924])

In [64]:

Total_surplus_generated = sum(y_pred_abs)
Total_surplus_generated

np.float64(154.07211685180664)